In [1]:
import re
import time
from datetime import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.delhisldc.org/Loaddata.aspx"
START_DATE = "02-04-2026"   # dd-mm-yyyy
END_DATE   = "11-06-2026"   # dd-mm-yyyy
OUT_FILE   = r"C:\Users\suhan\Desktop\Final Year Project\PowerDemand\DataSet_Training\delhi_sldc_5min_2022_2026.csv"

retry = Retry(total=5, backoff_factor=1, status_forcelist=[429,500,502,503,504], allowed_methods=["GET"])
adapter = HTTPAdapter(max_retries=retry)

session = requests.Session()
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

def clean_num(x):
    x = str(x).strip().replace(",", "")
    m = re.search(r"-?\d+(\.\d+)?", x)
    return float(m.group()) if m else None

def extract_timeslot(x):
    x = str(x).strip()
    m = re.search(r"\b([0-2]?\d:[0-5]\d)\b", x)
    return m.group(1) if m else None

def parse_day_html(html, date_ddmmyyyy):
    soup = BeautifulSoup(html, "lxml")
    rows_out = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = tr.find_all(["td", "th"])
            vals = [c.get_text(" ", strip=True) for c in cells]
            if len(vals) < 7:
                continue

            ts = extract_timeslot(vals[0])
            if not ts:
                continue

            nums = [clean_num(v) for v in vals[1:7]]
            if sum(v is not None for v in nums) < 4:
                continue

            row = {
                "Date": date_ddmmyyyy,
                "TimeSlot": ts,
                "DELHI": nums[0],
                "BRPL": nums[1],
                "BYPL": nums[2],
                "NDPL": nums[3],
                "NDMC": nums[4],
                "MES": nums[5],
            }
            rows_out.append(row)

    if not rows_out:
        return pd.DataFrame(columns=["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"])

    df = pd.DataFrame(rows_out).drop_duplicates(subset=["Date", "TimeSlot"], keep="first")
    df["sort_key"] = pd.to_datetime(df["Date"] + " " + df["TimeSlot"], format="%d/%m/%Y %H:%M", errors="coerce")
    df = df.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)
    return df

def fetch_one_day(d):
    mode = d.strftime("%d/%m/%Y")
    date_out = d.strftime("%d/%m/%Y")
    url = f"{BASE_URL}?mode={mode}"

    try:
        r = session.get(url, timeout=40)
        r.raise_for_status()
        df = parse_day_html(r.text, date_out)
        return df
    except Exception as e:
        print(f"[WARN] {mode} failed: {e}")
        return pd.DataFrame(columns=["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"])

start = datetime.strptime(START_DATE, "%d-%m-%Y")
end   = datetime.strptime(END_DATE, "%d-%m-%Y")
days = pd.date_range(start=start, end=end, freq="D")

all_parts = []
for d in tqdm(days, desc="Fetching"):
    part = fetch_one_day(d)
    if not part.empty:
        all_parts.append(part)
    time.sleep(0.2)

if not all_parts:
    raise RuntimeError("No rows extracted. Check connectivity / site blocking.")

final_df = pd.concat(all_parts, ignore_index=True)
final_df = final_df[["Date","TimeSlot","DELHI","BRPL","BYPL","NDPL","NDMC","MES"]]

print("First row:", final_df.iloc[0]["Date"], final_df.iloc[0]["TimeSlot"])
print("Last row:", final_df.iloc[-1]["Date"], final_df.iloc[-1]["TimeSlot"])
print("Total new rows:", len(final_df))

# Append without header
final_df.to_csv(OUT_FILE, mode="a", index=False, header=False)
print("Appended successfully!")

Fetching:   0%|          | 0/71 [00:00<?, ?it/s]

First row: 02/04/2026 00:00
Last row: 11/06/2026 23:40
Total new rows: 20302
Appended successfully!
